# 医院-药店-餐馆面板数据（2012-2019）

本 notebook 主要：

##1. 合并原始的医院数据和新增医院数据(远程桌面筛选增加得到），对医院数据进行清洗（手动训练+AI），得到全部医院数据
##2.根据经纬度定位和county.shp空间合并，spatial join 区县名称，并和工商数据进行比较区县差异
##3.对最终的医院数据进行经营方式（公立/私立）、政策实施年份匹配，查询
##4. 得到医院-年份面板：根据成立日期、注销日期、吊销日期判断医院在 2012-2019 各年年末是否存续，
##5.和大众点评中的医院数据相互印证（药店和医院都进行AI训练分类）
##6.得到医院-周边药店-年份数据：计算周边 1km、2km、3km、1-2km、2-3km 的真药店、连锁药店、餐馆数量。
##7. 得到区县-年份面板：聚合区县药店、连锁药店、餐馆、公立医院、民营医院数量及占比，并汇总医院附近的 POI 暴露量。


In [ ]:
import pandas as pd
from pathlib import Path

# =========================
# 合并数据
# 文件路径
# =========================
FILE1 = Path(r"D:\Hospital_Pharmacy program\data\hospital\hospital_public_policy_data.csv")

FILE2 = Path(r"D:\Hospital_Pharmacy program\data\hospital\07_added_hospital_public_private_policy.csv")

OUTPUT = Path(r"D:\Hospital_Pharmacy program\data\hospital\hospital_final_data.csv")


# =========================
# 自动识别编码
# =========================
def read_csv_auto(path):
    for enc in ["utf-8-sig", "utf-8", "gb18030", "gbk"]:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore", low_memory=False)


# =========================
# 读取数据
# =========================
df1 = read_csv_auto(FILE1)
df2 = read_csv_auto(FILE2)

print("=" * 60)
print("原始数据维度")
print("文件1 hospital_public_policy_data:", df1.shape)
print("文件2 07_added_hospital_public_private_policy:", df2.shape)
print("=" * 60)


# =========================
# 保留字段
# =========================
keep_cols = [
    "companyid",
    "ENTNAME",
    "经营范围",
    "注册地址",
    "所在行政区划",
    "省份",
    "地区",
    "百度_wgs84经度",
    "百度_wgs84纬度",
    "wgs_lon_true",
    "wgs_lat_true",
    "成立日期",
    "企业状态编码",
    "注销日期",
    "吊销日期",
    "score",
    "public",
    "policy_year",
]

# 缺失列自动补充
for col in keep_cols:
    if col not in df1.columns:
        df1[col] = pd.NA
    if col not in df2.columns:
        df2[col] = pd.NA

df1 = df1[keep_cols].copy()
df2 = df2[keep_cols].copy()

# 添加来源列，方便查看重复来自哪个文件
df1["source"] = "hospital_public_policy_data"
df2["source"] = "07_added_hospital_public_private_policy"


# =========================
# 纵向合并
# =========================
df = pd.concat([df1, df2], ignore_index=True)

print("\n" + "=" * 60)
print("纵向合并后数据维度")
print("合并后总行数:", df.shape[0])
print("合并后总列数:", df.shape[1])
print("=" * 60)


# =========================
# 打印重复情况
# =========================
dup_mask = df.duplicated(
    subset=["companyid", "ENTNAME"],
    keep=False
)

dup_df = (
    df.loc[dup_mask]
      .sort_values(["companyid", "ENTNAME", "source"])
)

dup_group_count = (
    dup_df[["companyid", "ENTNAME"]]
    .drop_duplicates()
    .shape[0]
)

print("\n" + "=" * 60)
print("重复情况统计")
print(f"重复记录总行数: {len(dup_df):,}")
print(f"重复医院数量 companyid + ENTNAME: {dup_group_count:,}")
print("=" * 60)

print("\n重复记录前50行：")
print(
    dup_df[
        [
            "companyid",
            "ENTNAME",
            "source",
            "经营范围",
            "注册地址",
            "所在行政区划",
            "省份",
            "地区",
            "wgs_lon_true",
            "wgs_lat_true",
            "成立日期",
            "企业状态编码",
            "注销日期",
            "吊销日期",
            "score",
            "public",
            "policy_year",
        ]
    ].head(50)
)

# =========================
# 去重
# 保留 hospital_public_policy_data 中的数据
# 因为 df1 放在 concat 前面，所以 keep='first'
# =========================
df_final = df.drop_duplicates(
    subset=["companyid", "ENTNAME"],
    keep="first"
).copy()

# 最终结果不保留 source
df_final = df_final[keep_cols]

print("\n" + "=" * 60)
print("去重后最终数据维度")
print("最终行数:", df_final.shape[0])
print("最终列数:", df_final.shape[1])
print("=" * 60)

print("\n最终数据前10行：")
print(df_final.head(10))


# =========================
# 保存最终文件
# =========================
df_final.to_csv(
    OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("\n最终文件保存完成：")
print(OUTPUT)

原始数据维度
文件1 hospital_public_policy_data: (38458, 37)
文件2 07_added_hospital_public_private_policy: (1994, 81)

纵向合并后数据维度
合并后总行数: 40452
合并后总列数: 19

重复情况统计
重复记录总行数: 8
重复医院数量 companyid + ENTNAME: 4

重复记录前50行：
       companyid       ENTNAME                       source  \
11223   16065978      宜春协和男科医院  hospital_public_policy_data   
11224   16065978      宜春协和男科医院  hospital_public_policy_data   
11221   82040500      宜春协和男科医院  hospital_public_policy_data   
11222   82040500      宜春协和男科医院  hospital_public_policy_data   
12121  910417083  山西省沁源县第二人民医院  hospital_public_policy_data   
12122  910417083  山西省沁源县第二人民医院  hospital_public_policy_data   
0      910618525  山西省沁源县第二人民医院  hospital_public_policy_data   
1      910618525  山西省沁源县第二人民医院  hospital_public_policy_data   

                                                    经营范围              注册地址  \
11223     内科、中医、泌尿、性功能障碍、生殖整形、前列腺、不孕不育诊疗。（有效期至2008年8月9日）  江西省宜春市袁州区文体路554号   
11224     内科、中医、泌尿、性功能障碍、生殖整形、前列腺、不孕不育诊疗。（有效期至2008年8月9日）  江西省宜春市袁州区文体路55

C:\Users\cwx13\AppData\Local\Temp\ipykernel_2936\148827425.py:82: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df1, df2], ignore_index=True)



最终文件保存完成：
D:\Hospital_Pharmacy program\data\hospital\hospital_final_data.csv


医院判断原则：
- 名称优先：`药房`、`药店`、`医院管理`、`医院投资`、`诊所`、`研究所`、`鉴定所` 等名称特征优先判定为非医院
-经营范围辅助
- AI 判断是否是医院，打分
—生成文档 D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification

-分类结果:完整分类：40,448 行
-hosp_real.csv：36,883 行(评分在50分以上)

**按经纬度空间匹配区县，并与工商行政区划比较**

使用 `hosp_real.csv` 中的 `wgs_lon_true` / `wgs_lat_true` 与 `counties19.shp` 做 spatial join，得到坐标运营地所在省、市、区县；再和 `所在行政区划` 对应的工商注册地省、市、区县进行比较。工商注册地先按县级 `PAC` 匹配，县级匹配不到时再用 `prefecture_boundaries` 的 `市代码` 补匹配到市级。输出保存在 `data/hospital/hosp_panel/hosp_classification`。

In [2]:
# =========================
# 坐标空间匹配区县，并与工商行政区划比较
# =========================
from pathlib import Path
import pandas as pd
import geopandas as gpd

ROOT = Path(r"D:\Hospital_Pharmacy program")
HOSP_REAL_PATH = ROOT / "data" / "hospital" / "hosp_panel" / "hosp_classification" / "hosp_real.csv"
CLASSIFICATION_DIR = HOSP_REAL_PATH.parent
COUNTY_SHP = ROOT / "data" / "county" / "controls" / "county_boundaries" / "counties19.shp"
PREF_DIR = ROOT / "data" / "county" / "controls" / "prefecture_boundaries"
PREF_SHP = next(p for p in PREF_DIR.glob("*.shp") if p.stat().st_size > 1000)
CLASSIFICATION_DIR.mkdir(parents=True, exist_ok=True)

spatial_output_path = CLASSIFICATION_DIR / "hosp_real_spatial_admin_matched.csv"
spatial_diff_path = CLASSIFICATION_DIR / "hosp_real_spatial_admin_diff.csv"
spatial_summary_path = CLASSIFICATION_DIR / "hosp_real_spatial_admin_summary.csv"

def read_csv_auto_local(path):
    last_error = None
    for enc in ["utf-8-sig", "utf-8", "gb18030", "gbk"]:
        try:
            return pd.read_csv(path, encoding=enc, dtype=str, low_memory=False)
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def code6(values):
    s = pd.Series(values)
    out = pd.Series(pd.NA, index=s.index, dtype="string")
    numeric = pd.to_numeric(s, errors="coerce")
    mask_num = numeric.notna()
    if mask_num.any():
        out.loc[mask_num] = numeric.loc[mask_num].round().astype("Int64").astype("string")
    text = s.astype("string").str.extract(r"(\d{6})", expand=False)
    out = out.fillna(text)
    out = out.astype("string").str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(6)
    return out.mask(out.isin(["", "<NA>", "nan", "None", "000000"]))


hospital_geo = read_csv_auto_local(HOSP_REAL_PATH)
hospital_geo.columns = hospital_geo.columns.str.strip()
hospital_geo["_row_id"] = range(len(hospital_geo))
hospital_geo["admin_code6"] = code6(hospital_geo["所在行政区划"])
hospital_geo["wgs_lon_true_num"] = pd.to_numeric(hospital_geo["wgs_lon_true"], errors="coerce")
hospital_geo["wgs_lat_true_num"] = pd.to_numeric(hospital_geo["wgs_lat_true"], errors="coerce")
hospital_geo["baidu_wgs_lon_num"] = pd.to_numeric(hospital_geo.get("百度_wgs84经度"), errors="coerce")
hospital_geo["baidu_wgs_lat_num"] = pd.to_numeric(hospital_geo.get("百度_wgs84纬度"), errors="coerce")
hospital_geo["has_valid_wgs_coord"] = (
    hospital_geo["wgs_lon_true_num"].between(70, 140)
    & hospital_geo["wgs_lat_true_num"].between(15, 55)
    & hospital_geo["wgs_lon_true_num"].ne(0)
    & hospital_geo["wgs_lat_true_num"].ne(0)
)
hospital_geo["has_valid_baidu_wgs_coord"] = (
    hospital_geo["baidu_wgs_lon_num"].between(70, 140)
    & hospital_geo["baidu_wgs_lat_num"].between(15, 55)
    & hospital_geo["baidu_wgs_lon_num"].ne(0)
    & hospital_geo["baidu_wgs_lat_num"].ne(0)
)
hospital_geo["wgs_coord_filled_from_baidu"] = (
    ~hospital_geo["has_valid_wgs_coord"]
    & hospital_geo["has_valid_baidu_wgs_coord"]
)
if "score" not in hospital_geo.columns:
    hospital_geo["score"] = pd.NA
hospital_geo.loc[hospital_geo["wgs_coord_filled_from_baidu"], "wgs_lon_true"] = hospital_geo.loc[
    hospital_geo["wgs_coord_filled_from_baidu"], "baidu_wgs_lon_num"
]
hospital_geo.loc[hospital_geo["wgs_coord_filled_from_baidu"], "wgs_lat_true"] = hospital_geo.loc[
    hospital_geo["wgs_coord_filled_from_baidu"], "baidu_wgs_lat_num"
]
hospital_geo.loc[hospital_geo["wgs_coord_filled_from_baidu"], "score"] = 6
hospital_geo["wgs_lon_true_num"] = pd.to_numeric(hospital_geo["wgs_lon_true"], errors="coerce")
hospital_geo["wgs_lat_true_num"] = pd.to_numeric(hospital_geo["wgs_lat_true"], errors="coerce")
hospital_geo["has_valid_wgs_coord"] = (
    hospital_geo["wgs_lon_true_num"].between(70, 140)
    & hospital_geo["wgs_lat_true_num"].between(15, 55)
    & hospital_geo["wgs_lon_true_num"].ne(0)
    & hospital_geo["wgs_lat_true_num"].ne(0)
)

county_boundaries = gpd.read_file(COUNTY_SHP).to_crs("EPSG:4326")
pref_boundaries = gpd.read_file(PREF_SHP).to_crs("EPSG:4326")

spatial_counties = county_boundaries.rename(
    columns={
        "PAC": "spatial_county_code",
        "NAME": "spatial_county",
        "省代码": "spatial_province_code",
        "省": "spatial_province",
        "市代码": "spatial_city_code",
        "市": "spatial_city",
        "类型": "spatial_county_type",
    }
)[
    [
        "spatial_county_code",
        "spatial_county",
        "spatial_province_code",
        "spatial_province",
        "spatial_city_code",
        "spatial_city",
        "spatial_county_type",
        "geometry",
    ]
].copy()
for col in ["spatial_county_code", "spatial_province_code", "spatial_city_code"]:
    spatial_counties[col] = code6(spatial_counties[col])

business_counties = county_boundaries.drop(columns="geometry").rename(
    columns={
        "PAC": "business_county_code",
        "NAME": "business_county",
        "省代码": "business_province_code",
        "省": "business_province",
        "市代码": "business_city_code",
        "市": "business_city",
        "类型": "business_county_type",
    }
)[
    [
        "business_county_code",
        "business_county",
        "business_province_code",
        "business_province",
        "business_city_code",
        "business_city",
        "business_county_type",
    ]
].drop_duplicates("business_county_code", keep="first")
for col in ["business_county_code", "business_province_code", "business_city_code"]:
    business_counties[col] = code6(business_counties[col])

business_prefs = pref_boundaries.drop(columns="geometry").rename(
    columns={
        "省代码": "business_pref_province_code",
        "省": "business_pref_province",
        "市代码": "business_pref_city_code",
        "市": "business_pref_city",
        "类型": "business_pref_type",
    }
)[
    [
        "business_pref_province_code",
        "business_pref_province",
        "business_pref_city_code",
        "business_pref_city",
        "business_pref_type",
    ]
].drop_duplicates("business_pref_city_code", keep="first")
for col in ["business_pref_province_code", "business_pref_city_code"]:
    business_prefs[col] = code6(business_prefs[col])
pref_city_codes = set(business_prefs["business_pref_city_code"].dropna())

def derive_pref_code(code):
    if pd.isna(code):
        return pd.NA
    code = str(code).strip()
    if len(code) != 6 or not code.isdigit():
        return pd.NA
    if code in pref_city_codes:
        return code
    candidate = code[:4] + "00"
    if candidate in pref_city_codes:
        return candidate
    if code[:2] in {"11", "12", "31", "50"}:
        candidate = code[:2] + "0000"
        if candidate in pref_city_codes:
            return candidate
    return pd.NA

hospital_geo["admin_pref_code6"] = hospital_geo["admin_code6"].map(derive_pref_code)

hospital_geo = hospital_geo.merge(
    business_counties,
    left_on="admin_code6",
    right_on="business_county_code",
    how="left",
)
hospital_geo = hospital_geo.merge(
    business_prefs,
    left_on="admin_pref_code6",
    right_on="business_pref_city_code",
    how="left",
)

hospital_geo["business_match_level"] = "unmatched"
hospital_geo.loc[hospital_geo["business_county_code"].notna(), "business_match_level"] = "county"
pref_fill = hospital_geo["business_county_code"].isna() & hospital_geo["business_pref_city_code"].notna()
hospital_geo.loc[pref_fill, "business_match_level"] = "prefecture"
hospital_geo["business_province_code"] = hospital_geo["business_province_code"].fillna(hospital_geo["business_pref_province_code"])
hospital_geo["business_province"] = hospital_geo["business_province"].fillna(hospital_geo["business_pref_province"])
hospital_geo["business_city_code"] = hospital_geo["business_city_code"].fillna(hospital_geo["business_pref_city_code"])
hospital_geo["business_city"] = hospital_geo["business_city"].fillna(hospital_geo["business_pref_city"])
hospital_geo["business_admin_type"] = hospital_geo["business_county_type"].fillna(hospital_geo["business_pref_type"])

valid_points = hospital_geo.loc[hospital_geo["has_valid_wgs_coord"]].copy()
hospital_points = gpd.GeoDataFrame(
    valid_points,
    geometry=gpd.points_from_xy(valid_points["wgs_lon_true_num"], valid_points["wgs_lat_true_num"]),
    crs="EPSG:4326",
)

spatial_joined = gpd.sjoin(
    hospital_points,
    spatial_counties,
    how="left",
    predicate="intersects",
)

spatial_cols = [
    "spatial_county_code",
    "spatial_county",
    "spatial_province_code",
    "spatial_province",
    "spatial_city_code",
    "spatial_city",
    "spatial_county_type",
]

spatial_joined = (
    spatial_joined.sort_values(["_row_id", "spatial_county_code"], na_position="last")
    .drop_duplicates("_row_id", keep="first")
    [["_row_id"] + spatial_cols]
)

hospital_spatial = hospital_geo.merge(spatial_joined, on="_row_id", how="left")

def code_diff(left, right):
    left = left.astype("string").str.strip()
    right = right.astype("string").str.strip()
    both = left.notna() & right.notna()
    return both & left.ne(right)


hospital_spatial["province_diff"] = code_diff(
    hospital_spatial["business_province_code"],
    hospital_spatial["spatial_province_code"],
)
hospital_spatial["city_diff"] = code_diff(
    hospital_spatial["business_city_code"],
    hospital_spatial["spatial_city_code"],
)
hospital_spatial["county_diff_comparable"] = (
    hospital_spatial["business_match_level"].eq("county")
    & hospital_spatial["business_county_code"].notna()
    & hospital_spatial["spatial_county_code"].notna()
)
hospital_spatial["county_diff"] = hospital_spatial["county_diff_comparable"] & code_diff(
    hospital_spatial["business_county_code"],
    hospital_spatial["spatial_county_code"],
)
hospital_spatial["any_admin_diff"] = hospital_spatial[
    ["province_diff", "city_diff", "county_diff"]
].any(axis=1)

hospital_spatial["spatial_match_status"] = "same_county_or_city"
hospital_spatial.loc[
    hospital_spatial["business_match_level"].eq("prefecture")
    & ~hospital_spatial["any_admin_diff"],
    "spatial_match_status",
] = "same_city_prefecture_registered"
hospital_spatial.loc[~hospital_spatial["has_valid_wgs_coord"], "spatial_match_status"] = "invalid_or_missing_coord"
hospital_spatial.loc[
    hospital_spatial["has_valid_wgs_coord"] & hospital_spatial["spatial_county_code"].isna(),
    "spatial_match_status",
] = "spatial_unmatched"
hospital_spatial.loc[
    hospital_spatial["has_valid_wgs_coord"]
    & hospital_spatial["spatial_county_code"].notna()
    & hospital_spatial["business_match_level"].eq("unmatched"),
    "spatial_match_status",
] = "business_admin_unmatched"
hospital_spatial.loc[hospital_spatial["any_admin_diff"], "spatial_match_status"] = "different"

diff_cols = [
    "companyid",
    "ENTNAME",
    "注册地址",
    "所在行政区划",
    "省份",
    "地区",
    "wgs_lon_true",
    "wgs_lat_true",
    "百度_wgs84经度",
    "百度_wgs84纬度",
    "wgs_coord_filled_from_baidu",
    "score",
    "public",
    "policy_year",
    "realness_score_100",
    "real_hospital_label",
    "is_likely_real_hospital",
    "admin_code6",
    "admin_pref_code6",
    "business_match_level",
    "business_province",
    "business_city",
    "business_county",
    "business_county_code",
    "business_pref_city_code",
    "business_admin_type",
    "spatial_province",
    "spatial_city",
    "spatial_county",
    "spatial_county_code",
    "province_diff",
    "city_diff",
    "county_diff_comparable",
    "county_diff",
    "any_admin_diff",
    "spatial_match_status",
]
diff_cols = [c for c in diff_cols if c in hospital_spatial.columns]
spatial_diff = hospital_spatial.loc[hospital_spatial["any_admin_diff"], diff_cols].copy()

summary_metrics = pd.DataFrame(
    {
        "metric": [
            "total_rows",
            "valid_wgs_coord",
            "baidu_coord_fill_for_invalid_wgs",
            "spatial_county_matched",
            "spatial_unmatched_with_valid_coord",
            "business_admin_county_matched",
            "business_admin_prefecture_matched",
            "business_admin_any_matched",
            "business_admin_unmatched",
            "province_diff",
            "city_diff",
            "county_diff_comparable",
            "county_diff",
            "any_admin_diff",
        ],
        "rows": [
            len(hospital_spatial),
            hospital_spatial["has_valid_wgs_coord"].sum(),
            hospital_spatial["wgs_coord_filled_from_baidu"].sum(),
            hospital_spatial["spatial_county_code"].notna().sum(),
            (hospital_spatial["has_valid_wgs_coord"] & hospital_spatial["spatial_county_code"].isna()).sum(),
            hospital_spatial["business_match_level"].eq("county").sum(),
            hospital_spatial["business_match_level"].eq("prefecture").sum(),
            hospital_spatial["business_match_level"].isin(["county", "prefecture"]).sum(),
            hospital_spatial["business_match_level"].eq("unmatched").sum(),
            hospital_spatial["province_diff"].sum(),
            hospital_spatial["city_diff"].sum(),
            hospital_spatial["county_diff_comparable"].sum(),
            hospital_spatial["county_diff"].sum(),
            hospital_spatial["any_admin_diff"].sum(),
        ],
    }
)
summary_metrics["share"] = summary_metrics["rows"] / len(hospital_spatial)

status_summary = (
    hospital_spatial["spatial_match_status"]
    .value_counts(dropna=False)
    .rename_axis("metric")
    .reset_index(name="rows")
)
status_summary["metric"] = "status_" + status_summary["metric"].astype(str)
status_summary["share"] = status_summary["rows"] / len(hospital_spatial)

spatial_summary = pd.concat([summary_metrics, status_summary], ignore_index=True)

def save_csv_or_updated(df, path):
    try:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        return path
    except PermissionError:
        fallback = path.with_name(f"{path.stem}_updated{path.suffix}")
        df.to_csv(fallback, index=False, encoding="utf-8-sig")
        print(f"目标文件被占用，已另存为: {fallback}")
        return fallback


actual_spatial_output_path = save_csv_or_updated(
    hospital_spatial.drop(columns=["_row_id"], errors="ignore"),
    spatial_output_path,
)
actual_spatial_diff_path = save_csv_or_updated(spatial_diff, spatial_diff_path)
actual_spatial_summary_path = save_csv_or_updated(spatial_summary, spatial_summary_path)

print(spatial_summary)
print("完整空间匹配结果（含运营地/注册地比较）:", actual_spatial_output_path)
print("运营地和注册地区县差异明细:", actual_spatial_diff_path)
print("空间匹配汇总:", actual_spatial_summary_path)
print("运营地和注册地存在差异的前20行:")
display(spatial_diff.head(20))

                                    metric   rows     share
0                               total_rows  36883  1.000000
1                          valid_wgs_coord  36883  1.000000
2         baidu_coord_fill_for_invalid_wgs    639  0.017325
3                   spatial_county_matched  36875  0.999783
4       spatial_unmatched_with_valid_coord      8  0.000217
5            business_admin_county_matched  28623  0.776049
6        business_admin_prefecture_matched   6118  0.165876
7               business_admin_any_matched  34741  0.941924
8                 business_admin_unmatched   2142  0.058076
9                            province_diff    503  0.013638
10                               city_diff    767  0.020795
11                  county_diff_comparable  28615  0.775832
12                             county_diff   2337  0.063363
13                          any_admin_diff   2509  0.068026
14              status_same_county_or_city  26278  0.712469
15  status_same_city_prefecture_register

,companyid,ENTNAME,注册地址,所在行政区划,省份,地区,wgs_lon_true,wgs_lat_true,百度_wgs84经度,百度_wgs84纬度,...,spatial_province,spatial_city,spatial_county,spatial_county_code,province_diff,city_diff,county_diff_comparable,county_diff,any_admin_diff,spatial_match_status
9,920799266,七台河市东方医院,黑龙江省七台河市桃山区桃山区大同街6号,230903.0,黑龙江省,七台河市,131.0562004360241,45.77185367205329,108.64152619397112,19.10245902981369,...,黑龙江省,七台河市,茄子河区,230904,False,False,True,True,True,different
17,921256237,七台河眼科医院,黑龙江省七台河市桃山区大同街223号,230903.0,黑龙江省,七台河市,130.5599508424594,45.74816888405468,130.8603856445681,45.772145662609326,...,黑龙江省,七台河市,勃利县,230921,False,False,True,True,True,different
20,920764213,七台河铁麒煤矿职工医院,黑龙江省七台河市茄子河区杨杨街,230904.0,黑龙江省,七台河市,131.02021269674077,45.76698906792702,130.8603856445681,45.772145662609326,...,黑龙江省,七台河市,桃山区,230903,False,False,True,True,True,different
35,921057543,七星关同德医院,七星关区千溪乡中屯村,520502.0,贵州省,毕节市,104.83013962273505,27.1281856898028,105.4711520562772,27.315639519219385,...,贵州省,毕节市,赫章县,520527,False,False,True,True,True,different
39,921057816,七星关奕安医院,大方县普宜收费站,520521.0,贵州省,毕节市,105.6634043664739,27.633285743712506,105.30172541388907,27.301743882976773,...,贵州省,毕节市,七星关区,520502,False,False,True,True,True,different
53,81067983,七星关燕骨悦医院,贵州省毕节市七星关区市东街道办环城北路原交警队办公楼,520502.0,贵州省,毕节市,106.0347607418272,27.032317153970503,105.28825559366798,27.31745682167189,...,贵州省,毕节市,黔西县,520522,False,False,True,True,True,different
112,920949889,三亚东南眼科医院,海南省三亚市天涯区文明路,460204.0,海南省,三亚市,109.5079162063344,18.254705078582017,109.5079162063344,18.254705078582017,...,海南省,三亚市,吉阳区,460203,False,False,True,True,True,different
115,43762790,三亚口腔医院,海南省三亚市天涯区胜利路望海花园丹海豪庭一楼东面三角区,460203.0,海南省,三亚市,109.49985533891645,18.24346659367112,112.3290965001466,16.8343388855706,...,海南省,三亚市,天涯区,460204,False,False,True,True,True,different
116,920949934,三亚安宁医院,海南省三亚市吉阳区三亚市田独镇榆红村,460203.0,海南省,三亚市,109.386502042272,18.31817010071285,109.5618503430826,18.271403379041395,...,海南省,三亚市,天涯区,460204,False,False,True,True,True,different
123,32918151,三亚诺尔医院,海南省三亚市榆亚路516号丽景家园A栋首层,460202.0,海南省,三亚市,109.54663193092802,18.236959043880915,109.50704695023651,18.23382698770358,...,海南省,三亚市,吉阳区,460203,False,False,True,True,True,different


复核注册地和运营地不同医院的地图坐标

筛选 `spatial_match_status == "different"` 的医院，重新调用百度/高德地图查询，转换为 WGS84 后和原 `wgs_lon_true` / `wgs_lat_true` 比较，并标记当前坐标是否可信。脚本支持缓存续跑，输出保存在 `hosp_classification`。

In [3]:
# 全量复核会调用地图 API；中断后重跑会读取缓存，跳过已完成记录。
# 如需先试跑，给脚本加参数：--limit 20
%run "D:/Hospital_Pharmacy program/code/python/recheck_different_coords.py" --min-interval 0.35

                              coord_recheck_flag  rows     share
0  current_coord_suspect_query_sources_agree_far  1143  0.455560
1           manual_review_query_sources_conflict   884  0.352332
2                current_coord_confirmed_le_500m   417  0.166202
3                   current_coord_near_500_1000m    65  0.025907
复核明细: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_different_coord_recheck.csv
合并标记: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_spatial_admin_matched_coord_rechecked.csv
复核汇总: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_different_coord_recheck_summary.csv
缓存: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_different_coord_recheck_cache.csv


根据复核结果生成最终坐标

`confirmed_le_500m` 和 `near_500_1000m` 保留原坐标；`suspect_query_sources_agree_far` 使用重新查询坐标替代；`manual_review_query_sources_conflict` 进一步做百度/高德逆地理编码和模糊匹配，匹配上哪个来源就选哪个来源坐标，未匹配的保留为手工查询。

In [4]:
# 仅 manual_review_query_sources_conflict 会额外调用逆地理编码 API；支持缓存续跑。
# 如需先试跑，给脚本加参数：--limit 20
%run "D:/Hospital_Pharmacy program/code/python/finalize_rechecked_coords.py" --min-interval 0.25

reverse fuzzy checked 25/884
reverse fuzzy checked 50/884


reverse fuzzy checked 75/884
reverse fuzzy checked 100/884


reverse fuzzy checked 125/884
reverse fuzzy checked 150/884


reverse fuzzy checked 175/884
reverse fuzzy checked 200/884


reverse fuzzy checked 225/884
reverse fuzzy checked 250/884


reverse fuzzy checked 275/884
reverse fuzzy checked 300/884


... (7 outputs left out)

reverse fuzzy checked 725/884
reverse fuzzy checked 750/884
reverse fuzzy checked 775/884


reverse fuzzy checked 800/884
reverse fuzzy checked 825/884


reverse fuzzy checked 850/884
reverse fuzzy checked 875/884
reverse fuzzy checked 884/884


                            final_coord_decision   rows     share
0                    not_different_keep_original  34374  0.931974
1                 replace_with_recheck_agree_far   1143  0.030990
2  manual_query_required_reverse_fuzzy_unmatched    607  0.016457
3                            keep_original_coord    482  0.013068
4         replace_with_gaode_reverse_fuzzy_match    147  0.003986
5         replace_with_baidu_reverse_fuzzy_match    130  0.003525
6                   manual_review_required_total    607  0.016457
差异样本最终坐标决策: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_different_coord_final_decision.csv
全量最终坐标表: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_spatial_admin_matched_coord_final.csv
最终决策汇总: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_coord_final_decision_summary.csv
逆地理编码缓存: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\ho

 生成最终清理版医院数据

删除 `省份`、`地区` 和行政区划代码等中间列；按最终坐标决策回写 `wgs_lon_true` / `wgs_lat_true`，需要手工查询的样本保留原坐标并标记；保留工商注册地名称字段、空间匹配名称字段和 `diff` 差异类型。

In [5]:
%run "D:/Hospital_Pharmacy program/code/python/build_hosp_real_final_clean.py"

                                               metric   rows     share
0                                          total_rows  36883  1.000000
1                                coord_replaced_total   1420  0.038500
2                         manual_coord_query_required    607  0.016457
3    final_coord_decision_not_different_keep_original  34374  0.931974
4   final_coord_decision_replace_with_recheck_agre...   1143  0.030990
5   final_coord_decision_manual_query_required_rev...    607  0.016457
6            final_coord_decision_keep_original_coord    482  0.013068
7   final_coord_decision_replace_with_gaode_revers...    147  0.003986
8   final_coord_decision_replace_with_baidu_revers...    130  0.003525
9                                           diff_same  33513  0.908630
10                                     diff_unmatched   2150  0.058292
11                                   diff_county_diff    842  0.022829
12                                 diff_province_diff    229  0.006209
13    

In [6]:
# 对 province_diff 医院进行省份归属核验；输出 review 明细和带标注的全量文件。
# 批量搜索结果会缓存；人工核验覆盖表保存在 hosp_real_province_diff_manual_overrides.csv。
%run "D:/Hospital_Pharmacy program/code/python/review_province_diff_web.py" --min-interval 0.8


processed 25/229


processed 50/229


processed 75/229


processed 100/229


processed 125/229


processed 150/229


processed 175/229


processed 200/229


processed 225/229


processed 229/229



province_diff review side summary:
    side  rows
business   150
 spatial    75
   other     4

province_diff review status summary:
province_review_side                          province_review_status  rows
            business                confirmed_by_manual_place_review    11
            business         confirmed_by_name_or_registered_address   137
            business                                confirmed_by_web     2
               other confirmed_other_province_by_manual_place_review     4
             spatial                confirmed_by_manual_place_review     7
             spatial         confirmed_by_name_or_registered_address    68

Saved detail: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_province_diff_review.csv
Saved full reviewed file: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_final_clean_province_reviewed.csv
Saved cache: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_c

In [7]:
# 对确认省份不等于当前 spatial_province 的医院，用重新查询的百度/高德 WGS84 坐标纠正。
# 候选坐标重新 spatial join 后，优先选择落在确认省份的百度坐标，否则选择高德坐标；随后重做全量 spatial join 和 diff。
%run "D:/Hospital_Pharmacy program/code/python/update_final_clean_with_province_coords.py"


province coordinate update summary
final rows: 36,883
targets confirmed_province != old spatial_province: 154
coordinates replaced: 131
targets not replaced: 23

update status:
province_coord_update_status
replaced_baidu_matches_confirmed_province               127
not_replaced_no_candidate_matches_confirmed_province     23
replaced_gaode_matches_confirmed_province                 4

coord source:
province_coord_update_source
baidu_recheck_wgs84    127
not_replaced            23
gaode_recheck_wgs84      4

new diff:
diff
same             33638
unmatched         2151
county_diff        845
city_diff          151
province_diff       98

deleted intermediate files:
D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_province_diff_review.csv
D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_final_clean_province_reviewed.csv
D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_classification\hosp_real_province_diff_web_

In [8]:
# 对 city_diff 医院确认城市并修正坐标。
# 名称/注册地址中的城市证据优先；若当前坐标不在确认城市，则优先使用落在确认城市的百度 WGS84 坐标，否则使用高德 WGS84 坐标。
%run "D:/Hospital_Pharmacy program/code/python/update_final_clean_with_city_coords.py"


city coordinate update summary
final rows: 36,883
city_diff targets before update: 151
coordinates replaced: 46
current coordinates kept as confirmed: 97
manual/not replaced: 8

update status:
city_coord_update_status
keep_current_coord_city_confirmed                   97
replaced_baidu_matches_confirmed_city               39
not_replaced_no_candidate_matches_confirmed_city     5
replaced_gaode_matches_confirmed_city                5
not_replaced_manual_review_city_unconfirmed          3
restored_original_wgs_matches_confirmed_city         2

coord source:
city_coord_update_source
original_wgs                            59
baidu_recheck_wgs84                     40
current_wgs                             17
baidu                                   11
gaode                                    9
not_replaced                             8
gaode_recheck_wgs84                      5
original_wgs_restored_city_confirmed     2

new diff:
diff
same             33678
unmatched         2151
county

In [9]:
# 根据 spatial join 结果，重新匹配政策实施年份。
# 使用 hosp_real_final_clean.csv 中的 spatial_province / spatial_city / spatial_county / spatial_county_type
# 与 policy_year.xlsx 的政策时间线匹配；保留原 policy_year，并输出新旧政策年份差异标注。
%run "D:/Hospital_Pharmacy program/code/python/rematch_policy_year_spatial.py"


spatial policy year rematch summary
input encoding: utf-8-sig
rows: 36,883

policy match level:
policy_match_level
province_all_scope              13570
province_county_scope            9193
province_city_scope              6449
exact_city                       3969
unmatched                        3152
special_city_except_hangzhou      411
exact_county                      139

policy year diff status:
policy_year_diff_status
same                       17360
changed                    16368
new_missing_old_matched     3151
old_missing_new_matched        3
both_missing                   1

changed old -> new top 20:
policy_year_original_before_spatial_rematch policy_year_spatial  rows
                                     2017.0                2015  6831
                                     2017.0                2016  2754
                                     2017.0                2014  1144
                                     2016.0                2013   721
                          

整理 医院-年份 面板数据

In [13]:

from pathlib import Path
import gc
import struct
import warnings

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)

BASE_DIR = Path(r"D:\Hospital_Pharmacy program")
HOSPITAL_FILE = BASE_DIR / "data" / "hospital" / "hosp_real_final_clean.csv"
PHARMACY_DIR = BASE_DIR / "data" / "pharmacy" / "pharm_clean"
RESTAURANT_DIR = BASE_DIR / "data" / "pharmacy" / "restaurant"
COUNTY_DBF = BASE_DIR / "data" / "county" / "controls" / "county_boundaries" / "counties19.dbf"

HOSP_YEAR_DIR = BASE_DIR / "data" / "hospital" / "hosp_panel" / "hosp_year"
HOSP_PANEL_DIR = BASE_DIR / "data" / "hospital_pharmacy" / "hosp_pharm_panel"
HOSP_INTERMEDIATE_DIR = HOSP_PANEL_DIR / "intermediate"
PHARM_NEAREST_PUBLIC_DIR = HOSP_INTERMEDIATE_DIR / "pharmacy_nearest_public_hospital"
COUNTY_PHARM_NEAREST_PUBLIC_FILE = HOSP_PANEL_DIR / "county_pharmacy_nearest_public_hospital_counts.csv"
HOSP_PHARM_NEAREST_PUBLIC_FILE = HOSP_PANEL_DIR / "hospital_pharmacy_nearest_public_hospital_counts.csv"

for d in [HOSP_YEAR_DIR, HOSP_PANEL_DIR, HOSP_INTERMEDIATE_DIR, PHARM_NEAREST_PUBLIC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2012, 2020))
RADIUS_METERS = (500, 1000, 2000, 3000)
RADIUS_LABELS = {500: "500m", 1000: "1km", 2000: "2km", 3000: "3km"}
COUNT_LABELS = ("500m", "1km", "2km", "3km", "1_2km", "2_3km")
PUBLIC_DISTANCE_THRESHOLDS = {"500m": 500, "1km": 1000, "2km": 2000}

COL_CREATED = "\u6210\u7acb\u65e5\u671f"
COL_CANCELLED = "\u6ce8\u9500\u65e5\u671f"
COL_REVOKED = "\u540a\u9500\u65e5\u671f"
COL_COUNTY = "\u6240\u5728\u884c\u653f\u533a\u5212"
COL_PUBLIC = "public"

EARTH_RADIUS_M = 6_371_008.8
RADIUS_CHORD = {r: 2 * np.sin(r / (2 * EARTH_RADIUS_M)) for r in RADIUS_METERS}

print("hospital source:", HOSPITAL_FILE)
print("pharmacy source:", PHARMACY_DIR)
print("restaurant source:", RESTAURANT_DIR)
print("hospital-year output:", HOSP_YEAR_DIR)
print("hospital-pharmacy output:", HOSP_PANEL_DIR)


hospital source: D:\Hospital_Pharmacy program\data\hospital\hosp_real_final_clean.csv
pharmacy source: D:\Hospital_Pharmacy program\data\pharmacy\pharm_clean
restaurant source: D:\Hospital_Pharmacy program\data\pharmacy\restaurant
hospital-year output: D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_year
hospital-pharmacy output: D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel


In [14]:

def read_csv_auto(path, usecols=None, nrows=None, dtype=None):
    path = Path(path)
    last_error = None
    for enc in ("utf-8-sig", "utf-8", "gb18030", "gbk"):
        try:
            return pd.read_csv(
                path,
                encoding=enc,
                low_memory=False,
                usecols=usecols,
                nrows=nrows,
                dtype=dtype,
            )
        except UnicodeDecodeError as exc:
            last_error = exc
    raise last_error


def write_csv_safely(df, path):
    path = Path(path)
    try:
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print("saved", path)
        return
    except PermissionError as exc:
        if not path.exists():
            raise
        tmp_path = path.with_name(f"{path.stem}.tmp{path.suffix}")
        df.to_csv(tmp_path, index=False, encoding="utf-8-sig")
        try:
            tmp_path.replace(path)
            print("replaced locked csv via temp file", path)
            return
        except PermissionError:
            tmp_path.unlink(missing_ok=True)
            print("skip locked existing csv:", path, repr(exc))


def normalize_county_code(values):
    s = pd.Series(values)
    out = pd.Series(pd.NA, index=s.index, dtype="string")
    numeric = pd.to_numeric(s, errors="coerce")
    mask_num = numeric.notna()
    if mask_num.any():
        out.loc[mask_num] = numeric.loc[mask_num].round().astype("Int64").astype("string")
    text = s.astype("string").str.extract(r"(\d{6})", expand=False)
    out = out.fillna(text)
    out = out.astype("string").str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(6)
    return out.mask(out.isin(["", "<NA>", "nan", "None", "000000"]))




def read_dbf_table(path):
    path = Path(path)
    rows = []
    with path.open("rb") as handle:
        header = handle.read(32)
        record_count = struct.unpack("<I", header[4:8])[0]
        header_length = struct.unpack("<H", header[8:10])[0]
        record_length = struct.unpack("<H", header[10:12])[0]
        fields = []
        while True:
            first_byte = handle.read(1)
            if first_byte == b"\r":
                break
            descriptor = first_byte + handle.read(31)
            raw_name = descriptor[:11].split(b"\x00", 1)[0]
            fields.append((raw_name.decode("utf-8", "replace"), descriptor[16]))
        handle.seek(header_length)
        for _ in range(record_count):
            record = handle.read(record_length)
            if not record or record[:1] == b"\x1a":
                break
            if record[:1] == b"*":
                continue
            offset = 1
            row = {}
            for name, length in fields:
                row[name] = record[offset:offset + length].decode("utf-8", "replace").strip()
                offset += length
            rows.append(row)
    return pd.DataFrame(rows)


def normalize_text(values):
    return pd.Series(values).astype("string").str.strip()


def build_spatial_county_code_map():
    raw = read_dbf_table(COUNTY_DBF)
    # counties19.dbf columns are: OBJECTID, PAC, NAME, province code, province,
    # city code, city, county type, Shape_Leng, Shape_Area. Use positions for
    # Chinese field names so this cell is robust to notebook/source encodings.
    county_map = pd.DataFrame({
        "spatial_county_code": raw["PAC"],
        "spatial_county": raw["NAME"],
        "spatial_province": raw.iloc[:, 4],
        "spatial_city": raw.iloc[:, 6],
        "spatial_county_type": raw.iloc[:, 7],
    })
    keys = ["spatial_province", "spatial_city", "spatial_county", "spatial_county_type"]
    county_map["spatial_county_code"] = normalize_county_code(county_map["spatial_county_code"])
    for col in keys:
        county_map[col] = normalize_text(county_map[col])
    county_map = county_map[~county_map["spatial_county_code"].str[:2].isin(["71", "81", "82"])].copy()
    county_map = county_map.drop_duplicates(keys, keep="first")
    if county_map.duplicated(keys).any():
        raise ValueError("spatial county name map has duplicate keys")
    return county_map

def attach_spatial_county_code(hospital):
    keys = ["spatial_province", "spatial_city", "spatial_county", "spatial_county_type"]
    out = hospital.drop(columns=["spatial_county_code"], errors="ignore").copy()
    for col in keys:
        if col not in out.columns:
            out[col] = pd.NA
        out[col] = normalize_text(out[col])

    county_map = build_spatial_county_code_map()
    out = out.merge(
        county_map,
        on=keys,
        how="left",
        validate="many_to_one",
    )

    missing = int(out["spatial_county_code"].isna().sum())
    print("spatial_county_code missing rows:", missing)
    return out


def parse_date(values):
    return pd.to_datetime(values, errors="coerce")


def to_bool_flag(values):
    s = pd.Series(values)
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).ne(0)
    txt = s.astype("string").str.strip().str.lower()
    true_values = {"true", "1", "yes", "y", "t", "\u662f", "\u8fde\u9501", "chain", "chains"}
    return txt.isin(true_values).fillna(False)


def public_private_flags(values):
    txt = pd.Series(values).astype("string").str.strip()
    public_flag = txt.eq("\u516c\u7acb") | txt.str.contains("\u516c\u7acb", na=False)
    private_flag = txt.eq("\u6c11\u8425") | txt.str.contains("\u6c11\u8425", na=False)
    return public_flag.fillna(False), private_flag.fillna(False)


def valid_lon_lat(df, lon_col="lon", lat_col="lat"):
    lon = pd.to_numeric(df[lon_col], errors="coerce")
    lat = pd.to_numeric(df[lat_col], errors="coerce")
    return lon.between(70, 140) & lat.between(15, 55)


def lonlat_to_unit_xyz(lon, lat):
    lon_rad = np.deg2rad(np.asarray(lon, dtype="float64"))
    lat_rad = np.deg2rad(np.asarray(lat, dtype="float64"))
    cos_lat = np.cos(lat_rad)
    return np.column_stack([
        cos_lat * np.cos(lon_rad),
        cos_lat * np.sin(lon_rad),
        np.sin(lat_rad),
    ])


def radius_count_matrix(origin_xyz, target_xyz, radii=RADIUS_METERS):
    n = len(origin_xyz)
    if n == 0:
        return {r: np.array([], dtype="int64") for r in radii}
    if target_xyz is None or len(target_xyz) == 0:
        return {r: np.zeros(n, dtype="int64") for r in radii}
    tree = cKDTree(target_xyz)
    return {
        r: tree.query_ball_point(origin_xyz, r=RADIUS_CHORD[r], return_length=True).astype("int64")
        for r in radii
    }


def add_ring_columns(df, prefix):
    df[f"{prefix}_1_2km"] = df[f"{prefix}_2km"] - df[f"{prefix}_1km"]
    df[f"{prefix}_2_3km"] = df[f"{prefix}_3km"] - df[f"{prefix}_2km"]
    return df


## 3. Build hospital-year panel from establishment/cancellation/revocation dates

In [15]:

def load_hospital_master():
    hospital = read_csv_auto(HOSPITAL_FILE)
    print("hospital raw shape:", hospital.shape)

    # hospital_id is the row number in hosp_real_final_clean.csv,
    # starting from 0; companyid is not used.
    hospital = hospital.reset_index(drop=True).copy()
    hospital["hospital_id"] = np.arange(len(hospital), dtype="int64")
    hospital = hospital.drop(columns=["companyid"], errors="ignore")

    hospital["registered_county_id"] = normalize_county_code(hospital[COL_COUNTY])
    hospital = attach_spatial_county_code(hospital)
    hospital["county_id"] = hospital["spatial_county_code"]
    hospital["established_date"] = parse_date(hospital[COL_CREATED])
    hospital["cancelled_date"] = parse_date(hospital[COL_CANCELLED])
    hospital["revoked_date"] = parse_date(hospital[COL_REVOKED])

    hospital["lon"] = pd.to_numeric(
        hospital["wgs_lon_true"],
        errors="coerce",
    )
    hospital["lat"] = pd.to_numeric(
        hospital["wgs_lat_true"],
        errors="coerce",
    )
    hospital["has_valid_coord"] = valid_lon_lat(
        hospital,
        "lon",
        "lat",
    )

    public_flag, private_flag = public_private_flags(
        hospital[COL_PUBLIC]
    )
    hospital["public_flag"] = public_flag.astype("int8")
    hospital["private_flag"] = private_flag.astype("int8")

    hospital["policy_year"] = pd.to_numeric(
        hospital.get("policy_year"),
        errors="coerce",
    ).astype("Int64")

    return hospital


MANUAL_DUPLICATE_KEEP_ID = 0

MANUAL_DUPLICATE_NAMES = {
    "\u6c81\u6c34\u53bf\u7b2c\u4e8c\u4eba\u6c11\u533b\u9662",
    "\u5c71\u897f\u7701\u6c81\u6e90\u53bf\u7b2c\u4e8c\u4eba\u6c11\u533b\u9662",
}


def remove_duplicate_hospitals_for_year(part, year):
    removed_parts = []

    manual_mask = (
        part["ENTNAME"]
        .astype("string")
        .isin(MANUAL_DUPLICATE_NAMES)
        & part["hospital_id"].ne(MANUAL_DUPLICATE_KEEP_ID)
    )

    if manual_mask.any():
        removed = part.loc[manual_mask].copy()
        removed["duplicate_reason"] = (
            "manual_qinshui_qinyuan_duplicate"
        )
        removed_parts.append(removed)

    clean = part.loc[~manual_mask].copy()

    lon_key = pd.to_numeric(
        clean["lon"],
        errors="coerce",
    ).round(7)

    lat_key = pd.to_numeric(
        clean["lat"],
        errors="coerce",
    ).round(7)

    valid_coord = lon_key.notna() & lat_key.notna()

    exact_mask = (
        valid_coord
        & clean.assign(
            _lon_key=lon_key,
            _lat_key=lat_key,
        ).duplicated(
            ["ENTNAME", "_lon_key", "_lat_key"],
            keep="first",
        )
    )

    if exact_mask.any():
        removed = clean.loc[exact_mask].copy()
        removed["duplicate_reason"] = (
            "duplicate_name_coord_same_year"
        )
        removed_parts.append(removed)

    clean = clean.loc[~exact_mask].copy()

    if removed_parts:
        removed_year = pd.concat(
            removed_parts,
            ignore_index=True,
        )
        removed_year["duplicate_year"] = year

        print(
            year,
            "duplicate hospital-year rows removed:",
            len(removed_year),
        )
    else:
        removed_year = pd.DataFrame()

    return clean, removed_year


def build_yearly_hospital_data(hospital):
    yearly_parts = []
    duplicate_removed_parts = []

    for year in YEARS:
        year_end = pd.Timestamp(
            year=year,
            month=12,
            day=31,
        )

        exists = (
            hospital["established_date"].notna()
            & (hospital["established_date"] <= year_end)
            & (
                hospital["cancelled_date"].isna()
                | (hospital["cancelled_date"] > year_end)
            )
            & (
                hospital["revoked_date"].isna()
                | (hospital["revoked_date"] > year_end)
            )
        )

        part = hospital.loc[exists].copy()

        part["year"] = year
        part["year_end"] = year_end
        part["exists_year_end"] = 1

        # 不在这里生成 post_policy

        part, removed_y = remove_duplicate_hospitals_for_year(
            part,
            year,
        )

        if len(removed_y):
            duplicate_removed_parts.append(removed_y)

        first_cols = [
            "hospital_id",
            "ENTNAME",
            "year",
            "year_end",
            "exists_year_end",
            "county_id",
            "spatial_county_code",
            "registered_county_id",
            "policy_year",
            COL_PUBLIC,
            "public_flag",
            "private_flag",
            "lon",
            "lat",
            "has_valid_coord",
            "established_date",
            "cancelled_date",
            "revoked_date",
        ]

        other_cols = [
            c
            for c in part.columns
            if c not in first_cols and c != "companyid"
        ]

        part = part[
            [c for c in first_cols if c in part.columns]
            + other_cols
        ]

        year_file = (
            HOSP_YEAR_DIR
            / f"hospital_year_{year}.csv"
        )

        write_csv_safely(part, year_file)

        print(year, "hospitals:", len(part))

        yearly_parts.append(part)

    if duplicate_removed_parts:
        duplicate_removed = pd.concat(
            duplicate_removed_parts,
            ignore_index=True,
        )

        duplicate_file = (
            HOSP_YEAR_DIR
            / "hospital_year_duplicates_removed_2012_2019.csv"
        )

        write_csv_safely(
            duplicate_removed,
            duplicate_file,
        )

        print(
            "duplicate hospital-year rows removed total:",
            len(duplicate_removed),
        )

    hospital_year_panel = pd.concat(
        yearly_parts,
        ignore_index=True,
    )

    panel_file = (
        HOSP_YEAR_DIR
        / "hospital_year_panel_2012_2019.csv"
    )

    write_csv_safely(
        hospital_year_panel,
        panel_file,
    )

    print(
        "hospital-year panel shape:",
        hospital_year_panel.shape,
    )

    return hospital_year_panel


hospital_master = load_hospital_master()
hospital_year_base = build_yearly_hospital_data(
    hospital_master
)

hospital_year_base.head()



hospital raw shape: (36827, 26)
spatial_county_code missing rows: 10
2012 duplicate hospital-year rows removed: 2
saved D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_year\hospital_year_2012.csv
2012 hospitals: 9847
2013 duplicate hospital-year rows removed: 2
saved D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_year\hospital_year_2013.csv
2013 hospitals: 11196
2014 duplicate hospital-year rows removed: 2
saved D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_year\hospital_year_2014.csv
2014 hospitals: 12824
2015 duplicate hospital-year rows removed: 2
saved D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_year\hospital_year_2015.csv
2015 hospitals: 15977
2016 duplicate hospital-year rows removed: 2
saved D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_year\hospital_year_2016.csv
2016 hospitals: 20538
2017 duplicate hospital-year rows removed: 3
saved D:\Hospital_Pharmacy program\data\hospital\hosp_panel\hosp_year\hospital_year_201

,hospital_id,ENTNAME,year,year_end,exists_year_end,county_id,spatial_county_code,registered_county_id,policy_year,public,public_flag,private_flag,lon,lat,has_valid_coord,established_date,cancelled_date,revoked_date,经营范围,注册地址,所在行政区划,wgs_lon_true,wgs_lat_true,成立日期,企业状态编码,注销日期,吊销日期,score,realness_score_100,business_province,business_city,business_county,business_match_level,business_admin_type,spatial_province,spatial_city,spatial_county,spatial_county_type,county_diff,diff
0,0,山西省沁源县第二人民医院,2012,2012-12-31,1,140431,140431,140431,2016,公立,1,0,112.309962,36.671081,True,2011-09-16,NaT,NaT,为人民身体健康提供医疗与护理保健服务；医疗与护理；医学教学；医学研究；卫生医疗人员培训；卫生...,沁源县机构编制委员会,140431.0,112.309962,36.671081,2011/9/16,正常,NaN,NaN,9,100,山西省,长治市,沁源县,county,县,山西省,长治市,沁源县,县,False,same
1,4,一六七医院,2012,2012-12-31,1,510112,510112,510112,2016,民营,0,1,104.156020,30.615586,True,2005-11-02,NaT,NaT,内科、外科、妇产科；妇科专业、儿科、口腔科、医学检验科、医学影像科、中医科,龙泉驿区大面街道陵川路1号,510112.0,104.156020,30.615586,2005/11/2,正常,NaN,NaN,10,83,四川省,成都市,龙泉驿区,county,市辖区,四川省,成都市,龙泉驿区,市辖区,False,same
2,7,七台河众仁医院,2012,2012-12-31,1,230903,230903,230900,2017,民营,0,1,131.011114,45.767837,True,2009-11-15,NaT,NaT,内科、外科、疼痛科、康复科、妇科、皮肤科、口腔科、麻醉科、老年病科、营养科、医学检验科、超声...,黑龙江省七台河市桃山区大同路54号,230900.0,131.011114,45.767837,2009/11/15,1,NaN,NaN,8,88,黑龙江省,七台河市,NaN,prefecture,地级市,黑龙江省,七台河市,桃山区,市辖区,False,same
3,8,七台河勃利种畜场职工医院,2012,2012-12-31,1,230902,230902,230902,2017,民营,0,1,131.024194,45.858387,True,2010-08-20,NaT,NaT,内科、外科、妇科、儿科、中医科、口腔科、眼科、耳鼻喉科、麻醉科、预防保健科、医学检验科、医学影像科,黑龙江省七台河市新兴区七台河市种畜场场部,230902.0,131.024194,45.858387,2010/8/20,注销,NaN,NaN,9,100,黑龙江省,七台河市,新兴区,county,市辖区,黑龙江省,七台河市,新兴区,市辖区,False,same
4,9,七台河市东方医院,2012,2012-12-31,1,230903,230903,230903,2017,民营,0,1,130.990395,45.767613,True,2009-10-27,NaT,NaT,内、外、妇、儿、眼、耳鼻喉、口腔、皮肤、医疗美容、中医、康复医学、麻醉、医学检验、医学影像科。,黑龙江省七台河市桃山区桃山区大同街6号,230903.0,130.990395,45.767613,2009/10/27,正常,NaN,NaN,9,88,黑龙江省,七台河市,桃山区,county,市辖区,黑龙江省,七台河市,桃山区,市辖区,False,same


## 4. Load yearly pharmacy and restaurant POI data

In [16]:

def poi_file(kind, year):
    if kind == "pharmacy":
        return PHARMACY_DIR / f"renew_pharmacy_dzdp_pharmacies_{year}.csv"
    if kind == "restaurant":
        return RESTAURANT_DIR / f"dzdp_restaurants_{year}.csv"
    raise ValueError(kind)


def load_poi_year(kind, year):
    path = poi_file(kind, year)
    if not path.exists():
        raise FileNotFoundError(path)

    wanted = {"year", "shop_id", "is_chains", "lon_wgs", "lat_wgs", "PAC"}
    if kind == "pharmacy":
        wanted.update({"is_real_pharmacy", "pharmacy_realness_score_100"})

    df = read_csv_auto(path, usecols=lambda c: c in wanted)
    df["lon"] = pd.to_numeric(df["lon_wgs"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat_wgs"], errors="coerce")
    df["county_id"] = normalize_county_code(df["PAC"])
    df = df.loc[valid_lon_lat(df, "lon", "lat")].copy()

    if "shop_id" in df.columns:
        df = df.drop_duplicates(subset=["shop_id"], keep="first").copy()

    df["chain_flag"] = to_bool_flag(df["is_chains"]).astype("int8") if "is_chains" in df.columns else 0

    if kind == "pharmacy":
        if "is_real_pharmacy" in df.columns:
            real_flag = pd.to_numeric(df["is_real_pharmacy"], errors="coerce").fillna(0).astype("int8")
        elif "pharmacy_realness_score_100" in df.columns:
            real_flag = pd.to_numeric(df["pharmacy_realness_score_100"], errors="coerce").gt(50).astype("int8")
        else:
            real_flag = pd.Series(1, index=df.index, dtype="int8")
        df = df.loc[real_flag.eq(1)].copy()

    df["year"] = year
    return df


for kind in ["pharmacy", "restaurant"]:
    sample = load_poi_year(kind, YEARS[0])
    print(kind, YEARS[0], sample.shape)
    print(sample.head(2))
    del sample
    gc.collect()


pharmacy 2012 (19528, 11)
   year  shop_id is_chains     lon_wgs    lat_wgs       PAC  \
0  2012  5632389        No  122.970009  41.101640  210303.0   
1  2012  5583270        No  122.934065  41.114782  210311.0   

   pharmacy_realness_score_100  is_real_pharmacy         lon        lat  \
0                           95                 1  122.970009  41.101640   
1                           95                 1  122.934065  41.114782   

   chain_flag  
0           0  
1           0  
restaurant 2012 (120999, 9)
   year  shop_id is_chains     lon_wgs    lat_wgs       PAC         lon  \
0  2012  5381126        No  102.226250  31.905978  513201.0  102.226250   
1  2012  5388890        No  102.747738  34.104397  513232.0  102.747738   

         lat  chain_flag  
0  31.905978           0  
1  34.104397           0  


## 5. Count nearby pharmacies, chain pharmacies, and restaurants

In [17]:

def empty_nearby_columns(out):
    for prefix in ["pharm", "chain_pharm", "resta"]:
        for label in COUNT_LABELS:
            out[f"{prefix}_{label}"] = np.nan
    return out


def compute_nearby_counts_for_year(hospital_panel, year):
    hp = hospital_panel.loc[hospital_panel["year"].eq(year)].copy()
    out = hp[["hospital_id", "year", "county_id", "lon", "lat", "has_valid_coord"]].copy()
    out = empty_nearby_columns(out)

    valid_h = hp["has_valid_coord"].fillna(False).astype(bool)
    if not valid_h.any():
        return out
    hosp_xyz = lonlat_to_unit_xyz(hp.loc[valid_h, "lon"], hp.loc[valid_h, "lat"])

    pharmacy = load_poi_year("pharmacy", year)
    pharmacy_xyz = lonlat_to_unit_xyz(pharmacy["lon"], pharmacy["lat"])
    chain = pharmacy.loc[pharmacy["chain_flag"].eq(1)]
    chain_xyz = lonlat_to_unit_xyz(chain["lon"], chain["lat"]) if len(chain) else np.empty((0, 3))

    pharmacy_counts = radius_count_matrix(hosp_xyz, pharmacy_xyz)
    chain_counts = radius_count_matrix(hosp_xyz, chain_xyz)
    for radius, label in RADIUS_LABELS.items():
        out.loc[valid_h, f"pharm_{label}"] = pharmacy_counts[radius]
        out.loc[valid_h, f"chain_pharm_{label}"] = chain_counts[radius]

    restaurant = load_poi_year("restaurant", year)
    restaurant_xyz = lonlat_to_unit_xyz(restaurant["lon"], restaurant["lat"])
    restaurant_counts = radius_count_matrix(hosp_xyz, restaurant_xyz)
    for radius, label in RADIUS_LABELS.items():
        out.loc[valid_h, f"resta_{label}"] = restaurant_counts[radius]

    for prefix in ["pharm", "chain_pharm", "resta"]:
        out = add_ring_columns(out, prefix)

    count_cols = [
        f"{prefix}_{label}"
        for prefix in ["pharm", "chain_pharm", "resta"]
        for label in COUNT_LABELS
    ]
    for col in count_cols:
        out[col] = out[col].astype("Int64")

    del pharmacy, chain, restaurant, pharmacy_xyz, chain_xyz, restaurant_xyz
    gc.collect()
    return out


nearby_parts = []
for year in YEARS:
    print("computing", year)
    nearby_y = compute_nearby_counts_for_year(hospital_year_base, year)
    year_file = HOSP_INTERMEDIATE_DIR / f"hospital_nearby_counts_{year}.csv"
    write_csv_safely(nearby_y, year_file)
    print("nearby shape:", nearby_y.shape)
    nearby_parts.append(nearby_y)

hospital_nearby = pd.concat(nearby_parts, ignore_index=True)
hospital_nearby_file = HOSP_INTERMEDIATE_DIR / "hospital_nearby_counts_2012_2019.csv"
write_csv_safely(hospital_nearby, hospital_nearby_file)
print("nearby panel shape:", hospital_nearby.shape)
hospital_nearby.head()


computing 2012
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\intermediate\hospital_nearby_counts_2012.csv
nearby shape: (9847, 24)
computing 2013
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\intermediate\hospital_nearby_counts_2013.csv
nearby shape: (11196, 24)
computing 2014
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\intermediate\hospital_nearby_counts_2014.csv
nearby shape: (12824, 24)
computing 2015
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\intermediate\hospital_nearby_counts_2015.csv
nearby shape: (15977, 24)
computing 2016
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\intermediate\hospital_nearby_counts_2016.csv
nearby shape: (20538, 24)
computing 2017
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\intermediate\hospital_nearby_counts_2017.csv
nearby shape: (23946, 24)
computing 2018
saved D:\Hospital_Ph

,hospital_id,year,county_id,lon,lat,has_valid_coord,pharm_500m,pharm_1km,pharm_2km,pharm_3km,pharm_1_2km,pharm_2_3km,chain_pharm_500m,chain_pharm_1km,chain_pharm_2km,chain_pharm_3km,chain_pharm_1_2km,chain_pharm_2_3km,resta_500m,resta_1km,resta_2km,resta_3km,resta_1_2km,resta_2_3km
0,0,2012,140431,112.309962,36.671081,True,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,4,2012,510112,104.156020,30.615586,True,0,0,1,4,1,3,0,0,0,3,0,3,1,2,2,5,0,3
2,7,2012,230903,131.011114,45.767837,True,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1
3,8,2012,230902,131.024194,45.858387,True,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,9,2012,230903,130.990395,45.767613,True,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0


## 6. Count unique pharmacies assigned to the nearest same-county public hospital

These variables are built from the pharmacy side. Each pharmacy is matched to at most one active public hospital in the same county and year, so a pharmacy near multiple hospitals is counted only once. The code produces both county-year counts and hospital-year counts. For the hospital panel, an eligible public hospital with no assigned pharmacy is coded as zero; private hospitals and public hospitals without valid coordinates remain missing.

In [ ]:
def pharmacy_nearest_public_counts_for_year(hospital_panel, year):
    pharmacy = load_poi_year("pharmacy", year)
    pharmacy = pharmacy.dropna(subset=["shop_id", "county_id"]).copy()
    pharmacy = pharmacy.drop_duplicates("shop_id", keep="first")

    pharmacy_level = pharmacy[["shop_id", "year", "county_id", "lon", "lat"]].copy()
    pharmacy_level["nearest_public_hospital_id"] = pd.Series(pd.NA, index=pharmacy_level.index, dtype="Int64")
    pharmacy_level["nearest_public_hospital_m"] = np.nan
    pharmacy_level["has_same_county_public_hospital"] = 0

    public_hospitals = hospital_panel.loc[
        hospital_panel["year"].eq(year)
        & hospital_panel["public_flag"].eq(1)
        & hospital_panel["has_valid_coord"].fillna(False).astype(bool)
        & hospital_panel["county_id"].notna(),
        ["hospital_id", "county_id", "lon", "lat"],
    ].copy()

    public_by_county = {
        county_id: part.reset_index(drop=True)
        for county_id, part in public_hospitals.groupby("county_id", sort=False)
    }

    for county_id, pharmacy_index in pharmacy_level.groupby("county_id", sort=False).groups.items():
        county_hospitals = public_by_county.get(county_id)
        if county_hospitals is None or county_hospitals.empty:
            continue

        pharmacy_xyz = lonlat_to_unit_xyz(
            pharmacy_level.loc[pharmacy_index, "lon"],
            pharmacy_level.loc[pharmacy_index, "lat"],
        )
        hospital_xyz = lonlat_to_unit_xyz(county_hospitals["lon"], county_hospitals["lat"])
        nearest_chord, nearest_position = cKDTree(hospital_xyz).query(pharmacy_xyz, k=1)
        nearest_chord = np.clip(np.asarray(nearest_chord, dtype="float64"), 0.0, 2.0)
        nearest_distance_m = 2 * EARTH_RADIUS_M * np.arcsin(nearest_chord / 2)
        nearest_hospital_id = county_hospitals.iloc[np.asarray(nearest_position)]["hospital_id"].to_numpy()

        pharmacy_level.loc[pharmacy_index, "nearest_public_hospital_m"] = nearest_distance_m
        pharmacy_level.loc[pharmacy_index, "nearest_public_hospital_id"] = nearest_hospital_id
        pharmacy_level.loc[pharmacy_index, "has_same_county_public_hospital"] = 1

    for label, threshold_m in PUBLIC_DISTANCE_THRESHOLDS.items():
        within = pharmacy_level["nearest_public_hospital_m"].le(threshold_m)
        pharmacy_level[f"nearest_pub_le{label}"] = within.astype("int8")
        pharmacy_level[f"nearest_pub_gt{label}"] = (~within).astype("int8")

    count_aggregations = {
        "pharm_nearest_pub_total": ("shop_id", "nunique"),
        "pharm_has_same_county_pub": ("has_same_county_public_hospital", "sum"),
    }
    for label in PUBLIC_DISTANCE_THRESHOLDS:
        count_aggregations[f"pharm_nearest_pub_le{label}"] = (f"nearest_pub_le{label}", "sum")
        count_aggregations[f"pharm_nearest_pub_gt{label}"] = (f"nearest_pub_gt{label}", "sum")

    county_counts = pharmacy_level.groupby(["county_id", "year"], as_index=False).agg(**count_aggregations)

    for label in PUBLIC_DISTANCE_THRESHOLDS:
        assert (
            county_counts[f"pharm_nearest_pub_le{label}"]
            + county_counts[f"pharm_nearest_pub_gt{label}"]
            == county_counts["pharm_nearest_pub_total"]
        ).all()
    assert (county_counts["pharm_nearest_pub_le500m"] <= county_counts["pharm_nearest_pub_le1km"]).all()
    assert (county_counts["pharm_nearest_pub_le1km"] <= county_counts["pharm_nearest_pub_le2km"]).all()

    assigned = pharmacy_level.dropna(subset=["nearest_public_hospital_id"]).copy()
    hospital_aggregations = {
        "pharm_nearest_pub_total": ("shop_id", "nunique"),
    }
    for label in PUBLIC_DISTANCE_THRESHOLDS:
        hospital_aggregations[f"pharm_nearest_pub_le{label}"] = (f"nearest_pub_le{label}", "sum")
        hospital_aggregations[f"pharm_nearest_pub_gt{label}"] = (f"nearest_pub_gt{label}", "sum")

    assigned_counts = (
        assigned.groupby(["nearest_public_hospital_id", "year"], as_index=False)
        .agg(**hospital_aggregations)
        .rename(columns={"nearest_public_hospital_id": "hospital_id"})
    )
    hospital_counts = public_hospitals[["hospital_id"]].drop_duplicates().copy()
    hospital_counts["year"] = year
    hospital_counts = hospital_counts.merge(
        assigned_counts,
        on=["hospital_id", "year"],
        how="left",
        validate="one_to_one",
    )
    hospital_count_cols = list(hospital_aggregations)
    hospital_counts[hospital_count_cols] = (
        hospital_counts[hospital_count_cols].fillna(0).astype("int64")
    )
    for label in PUBLIC_DISTANCE_THRESHOLDS:
        assert (
            hospital_counts[f"pharm_nearest_pub_le{label}"]
            + hospital_counts[f"pharm_nearest_pub_gt{label}"]
            == hospital_counts["pharm_nearest_pub_total"]
        ).all()
    assert (hospital_counts["pharm_nearest_pub_le500m"] <= hospital_counts["pharm_nearest_pub_le1km"]).all()
    assert (hospital_counts["pharm_nearest_pub_le1km"] <= hospital_counts["pharm_nearest_pub_le2km"]).all()

    pharmacy_level_file = PHARM_NEAREST_PUBLIC_DIR / f"pharmacy_nearest_public_hospital_{year}.csv"
    write_csv_safely(pharmacy_level, pharmacy_level_file)
    return county_counts, hospital_counts


county_nearest_public_parts = []
hospital_nearest_public_parts = []
for year in YEARS:
    print("computing pharmacy-to-nearest-public-hospital distance", year)
    county_counts, hospital_counts = pharmacy_nearest_public_counts_for_year(hospital_year_base, year)
    county_nearest_public_parts.append(county_counts)
    hospital_nearest_public_parts.append(hospital_counts)

county_pharmacy_nearest_public = pd.concat(county_nearest_public_parts, ignore_index=True)
assert not county_pharmacy_nearest_public.duplicated(["county_id", "year"]).any()
write_csv_safely(county_pharmacy_nearest_public, COUNTY_PHARM_NEAREST_PUBLIC_FILE)
print("saved", COUNTY_PHARM_NEAREST_PUBLIC_FILE, county_pharmacy_nearest_public.shape)

hospital_pharmacy_nearest_public = pd.concat(hospital_nearest_public_parts, ignore_index=True)
assert not hospital_pharmacy_nearest_public.duplicated(["hospital_id", "year"]).any()
write_csv_safely(hospital_pharmacy_nearest_public, HOSP_PHARM_NEAREST_PUBLIC_FILE)
print("saved", HOSP_PHARM_NEAREST_PUBLIC_FILE, hospital_pharmacy_nearest_public.shape)
hospital_pharmacy_nearest_public.head()


## 7. Merge hospital-year panel with nearby POI counts

In [19]:

nearby_value_cols = [
    c for c in hospital_nearby.columns
    if c not in {"hospital_id", "year", "county_id", "lon", "lat", "has_valid_coord"}
]

hospital_panel = hospital_year_base.merge(
    hospital_nearby[["hospital_id", "year"] + nearby_value_cols],
    on=["hospital_id", "year"],
    how="left",
    validate="one_to_one",
)

nearest_public_value_cols = [
    c for c in hospital_pharmacy_nearest_public.columns
    if c not in {"hospital_id", "year"}
]
hospital_panel = hospital_panel.merge(
    hospital_pharmacy_nearest_public[["hospital_id", "year"] + nearest_public_value_cols],
    on=["hospital_id", "year"],
    how="left",
    validate="one_to_one",
)

hospital_panel = hospital_panel.sort_values(["hospital_id", "year"]).reset_index(drop=True)
hospital_panel = hospital_panel.drop(columns=["companyid"], errors="ignore")





加入夜间灯光数据


In [21]:
NTL_WIDE_FILE = BASE_DIR / "data" / "county" / "controls" / "gee_ntl" / "hospital_ntl_wide_2012_2019.csv"
HOSP_PANEL_FINAL_CSV = HOSP_PANEL_DIR / "hospital_pharm_panel.csv"
HOSP_PANEL_FINAL_DTA = HOSP_PANEL_DIR / "hospital_pharm_panel.dta"

import re

DTA_RENAME_MAP = {
    "经营范围": "business_scope",
    "注册地址": "reg_address",
    "企业状态编码": "enterprise_status_code",
    "ENTNAME": "entname",
}

STATA_VARIABLE_LABELS = {
    "hospital_id": "Hospital ID",
    "entname": "Hospital name",
    "year": "Calendar year",
    "county_id": "2019 county administrative code",
    "policy_year": "Local public-hospital reform year",
    "public": "Hospital ownership type",
    "public_flag": "Public hospital indicator",
    "private_flag": "Private hospital indicator",
    "lon": "Hospital longitude, WGS84",
    "lat": "Hospital latitude, WGS84",
    "established_date": "Hospital establishment date",
    "cancelled_date": "Hospital cancellation date",
    "revoked_date": "Hospital license revocation date",
    "business_scope": "Registered business scope",
    "reg_address": "Registered address",
    "enterprise_status_code": "Enterprise status code",
    "score": "Hospital identification score",
    "realness_score_100": "Hospital identification score, 0-100",
    "spatial_province": "Province assigned from hospital coordinates",
    "spatial_city": "Prefecture assigned from hospital coordinates",
    "spatial_county": "County assigned from hospital coordinates",
    "spatial_county_type": "County type assigned from coordinates",
}
for scope, scope_label in {
    "500m": "within 500 m",
    "1km": "within 1 km",
    "2km": "within 2 km",
    "3km": "within 3 km",
    "1_2km": "in the 1-2 km ring",
    "2_3km": "in the 2-3 km ring",
}.items():
    STATA_VARIABLE_LABELS[f"pharm_{scope}"] = f"Pharmacies {scope_label} of hospital"
    STATA_VARIABLE_LABELS[f"chain_pharm_{scope}"] = f"Chain pharmacies {scope_label} of hospital"
    STATA_VARIABLE_LABELS[f"resta_{scope}"] = f"Restaurants {scope_label} of hospital"
    STATA_VARIABLE_LABELS[f"ln_ntl_mean_{scope}"] = f"Ln mean nighttime lights plus one, {scope_label}"

STATA_VARIABLE_LABELS["pharm_nearest_pub_total"] = "Pharmacies assigned to nearest same-county public hospital"
for scope, scope_label in {"500m": "500 m", "1km": "1 km", "2km": "2 km"}.items():
    STATA_VARIABLE_LABELS[f"pharm_nearest_pub_le{scope}"] = f"Assigned pharmacies within {scope_label}"
    STATA_VARIABLE_LABELS[f"pharm_nearest_pub_gt{scope}"] = f"Assigned pharmacies beyond {scope_label}"


def stata_safe_name(name, used):
    candidate = DTA_RENAME_MAP.get(name)
    if candidate is None:
        candidate = re.sub(r"[^A-Za-z0-9_]", "_", name)
        candidate = re.sub(r"_+", "_", candidate).strip("_").lower()
        if not candidate:
            candidate = "var"
        if not re.match(r"[A-Za-z_]", candidate):
            candidate = "v_" + candidate
        candidate = candidate[:32]
    base_candidate = candidate[:32]
    i = 1
    while candidate.lower() in used:
        suffix = f"_{i}"
        candidate = base_candidate[:32 - len(suffix)] + suffix
        i += 1
    used.add(candidate.lower())
    return candidate


def truncate_utf8_for_stata(value, max_bytes=2000):
    if pd.isna(value):
        return ""
    text = str(value)
    raw = text.encode("utf-8")
    if len(raw) <= max_bytes:
        return text
    return raw[:max_bytes].decode("utf-8", errors="ignore")


def export_stata_panel(df, dta_path):
    original_cols = list(df.columns)
    used = set()
    dta_cols = [stata_safe_name(c, used) for c in original_cols]
    dta_df = df.copy()
    dta_df.columns = dta_cols

    for col in dta_df.columns:
        if dta_df[col].dtype == bool:
            dta_df[col] = dta_df[col].astype("int8")
    for col in dta_df.select_dtypes(include=["object", "string"]).columns:
        dta_df[col] = dta_df[col].map(truncate_utf8_for_stata).astype(str)

    num_cols = dta_df.select_dtypes(include=["number"]).columns
    if len(num_cols):
        dta_df[num_cols] = dta_df[num_cols].replace([float("inf"), float("-inf")], pd.NA)

    map_path = dta_path.with_name(f"{dta_path.stem}_dta_variable_map.csv")
    pd.DataFrame({"original_column": original_cols, "dta_variable": dta_cols}).to_csv(
        map_path, index=False, encoding="utf-8-sig"
    )
    if dta_path.exists():
        dta_path.unlink()
    variable_labels = {
        column: STATA_VARIABLE_LABELS[column]
        for column in dta_df.columns
        if column in STATA_VARIABLE_LABELS
    }
    missing_labels = sorted(set(dta_df.columns) - set(variable_labels))
    assert not missing_labels, f"Variables without Stata labels: {missing_labels}"
    assert all(len(label) <= 80 for label in variable_labels.values())

    dta_df.to_stata(
        dta_path,
        write_index=False,
        version=118,
        data_label="hospital pharmacy panel",
        convert_strl=None,
        variable_labels=variable_labels,
    )
    print("saved", dta_path)
    print("saved", map_path)


ntl_panel = read_csv_auto(NTL_WIDE_FILE)
ntl_panel["hospital_id"] = pd.to_numeric(ntl_panel["hospital_id"], errors="coerce").astype("Int64")
ntl_panel["year"] = pd.to_numeric(ntl_panel["year"], errors="coerce").astype("Int64")
ntl_panel = ntl_panel[ntl_panel["year"].isin(YEARS)].dropna(subset=["hospital_id", "year"]).copy()
assert not ntl_panel.duplicated(["hospital_id", "year"]).any()

ntl_value_cols = [c for c in ntl_panel.columns if c.startswith("ln_ntl_mean_")]
required_ntl_cols = ["ln_ntl_mean_500m", "ln_ntl_mean_1km", "ln_ntl_mean_2km", "ln_ntl_mean_3km"]
missing_ntl_cols = [c for c in required_ntl_cols if c not in ntl_panel.columns]
if missing_ntl_cols:
    raise ValueError(f"missing NTL columns: {missing_ntl_cols}")

hospital_panel_for_ntl = hospital_panel.copy()
hospital_panel_for_ntl["hospital_id"] = pd.to_numeric(hospital_panel_for_ntl["hospital_id"], errors="coerce").astype("Int64")
hospital_panel_for_ntl["year"] = pd.to_numeric(hospital_panel_for_ntl["year"], errors="coerce").astype("Int64")
hospital_panel_for_ntl = hospital_panel_for_ntl.drop(columns=[c for c in ntl_value_cols if c in hospital_panel_for_ntl.columns], errors="ignore")

hospital_pharm_panel = hospital_panel_for_ntl.merge(
    ntl_panel[["hospital_id", "year"] + ntl_value_cols],
    on=["hospital_id", "year"],
    how="left",
    validate="one_to_one",
)

assert len(hospital_pharm_panel) == len(hospital_panel)
assert "pharm_500m" in hospital_pharm_panel.columns
assert "chain_pharm_500m" in hospital_pharm_panel.columns
assert "resta_500m" in hospital_pharm_panel.columns
assert "ln_ntl_mean_500m" in hospital_pharm_panel.columns

remove_columns = [
    "year_end",
    "exists_year_end",
    "spatial_county_code",
    "registered_county_id",
    "has_valid_coord",
    COL_COUNTY,
    "wgs_lon_true",
    "wgs_lat_true",
    COL_CREATED,
    COL_CANCELLED,
    COL_REVOKED,
    "business_province",
    "business_city",
    "business_county",
    "business_match_level",
    "business_admin_type",
    "county_diff",
    "diff",
]
hospital_pharm_panel = hospital_pharm_panel.drop(columns=remove_columns, errors="ignore")

ntl_non_final_cols = [
    c for c in hospital_pharm_panel.columns
    if c.startswith(("ntl_mean_", "ntl_sum_", "ntl_pixels_", "ln_ntl_sum_"))
]
hospital_pharm_panel = hospital_pharm_panel.drop(columns=ntl_non_final_cols, errors="ignore")

unexpected_removed_cols = [c for c in remove_columns if c in hospital_pharm_panel.columns]
unexpected_ntl_cols = [
    c for c in hospital_pharm_panel.columns
    if c.startswith(("ntl_mean_", "ntl_sum_", "ntl_pixels_", "ln_ntl_sum_"))
]
assert not unexpected_removed_cols, unexpected_removed_cols
assert not unexpected_ntl_cols, unexpected_ntl_cols

missing_ln_ntl_500m = int(hospital_pharm_panel["ln_ntl_mean_500m"].isna().sum())
write_csv_safely(hospital_pharm_panel, HOSP_PANEL_FINAL_CSV)
export_stata_panel(hospital_pharm_panel, HOSP_PANEL_FINAL_DTA)

print("saved", HOSP_PANEL_FINAL_CSV, hospital_pharm_panel.shape)
print("saved", HOSP_PANEL_FINAL_DTA)
print("missing ln_ntl_mean_500m:", missing_ln_ntl_500m)
hospital_pharm_panel.head()


saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\hospital_pharm_panel.csv
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\hospital_pharm_panel.dta
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\hospital_pharm_panel_dta_variable_map.csv
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\hospital_pharm_panel.csv (147783, 46)
saved D:\Hospital_Pharmacy program\data\hospital_pharmacy\hosp_pharm_panel\hospital_pharm_panel.dta
missing ln_ntl_mean_500m: 0


,hospital_id,ENTNAME,year,county_id,policy_year,public,public_flag,private_flag,lon,lat,established_date,cancelled_date,revoked_date,经营范围,注册地址,企业状态编码,score,realness_score_100,spatial_province,spatial_city,spatial_county,spatial_county_type,pharm_500m,pharm_1km,pharm_2km,pharm_3km,pharm_1_2km,pharm_2_3km,chain_pharm_500m,chain_pharm_1km,chain_pharm_2km,chain_pharm_3km,chain_pharm_1_2km,chain_pharm_2_3km,resta_500m,resta_1km,resta_2km,resta_3km,resta_1_2km,resta_2_3km,ln_ntl_mean_500m,ln_ntl_mean_1km,ln_ntl_mean_2km,ln_ntl_mean_3km,ln_ntl_mean_1_2km,ln_ntl_mean_2_3km
0,0,山西省沁源县第二人民医院,2012,140431,2016,公立,1,0,112.309962,36.671081,2011-09-16,NaT,NaT,为人民身体健康提供医疗与护理保健服务；医疗与护理；医学教学；医学研究；卫生医疗人员培训；卫生...,沁源县机构编制委员会,正常,9,100,山西省,长治市,沁源县,县,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.741630,0.721094,0.672987,0.483015,0.672987,0.483015
1,0,山西省沁源县第二人民医院,2013,140431,2016,公立,1,0,112.309962,36.671081,2011-09-16,NaT,NaT,为人民身体健康提供医疗与护理保健服务；医疗与护理；医学教学；医学研究；卫生医疗人员培训；卫生...,沁源县机构编制委员会,正常,9,100,山西省,长治市,沁源县,县,1,1,1,1,0,0,0,0,0,0,0,0,2,4,4,4,0,0,0.849503,0.955879,0.873684,0.590602,0.873684,0.590602
2,0,山西省沁源县第二人民医院,2014,140431,2016,公立,1,0,112.309962,36.671081,2011-09-16,NaT,NaT,为人民身体健康提供医疗与护理保健服务；医疗与护理；医学教学；医学研究；卫生医疗人员培训；卫生...,沁源县机构编制委员会,正常,9,100,山西省,长治市,沁源县,县,3,3,3,3,0,0,0,0,0,0,0,0,2,4,4,4,0,0,0.751858,0.907913,0.783833,0.526327,0.783833,0.526327
3,0,山西省沁源县第二人民医院,2015,140431,2016,公立,1,0,112.309962,36.671081,2011-09-16,NaT,NaT,为人民身体健康提供医疗与护理保健服务；医疗与护理；医学教学；医学研究；卫生医疗人员培训；卫生...,沁源县机构编制委员会,正常,9,100,山西省,长治市,沁源县,县,3,3,3,3,0,0,0,0,0,0,0,0,2,4,4,4,0,0,0.987968,1.021109,0.895571,0.612371,0.895571,0.612371
4,0,山西省沁源县第二人民医院,2016,140431,2016,公立,1,0,112.309962,36.671081,2011-09-16,NaT,NaT,为人民身体健康提供医疗与护理保健服务；医疗与护理；医学教学；医学研究；卫生医疗人员培训；卫生...,沁源县机构编制委员会,正常,9,100,山西省,长治市,沁源县,县,4,4,4,4,0,0,0,0,0,0,0,0,4,5,6,6,1,0,0.813661,0.782397,0.636486,0.407994,0.636486,0.407994
